## final_project_part_3.ipynb
### DSA4323/5323
### University of Oklahoma
### <font color="red">This part of the final is for graduate students only!</font>
This notebook is the thrid of three of the final project for Ethics and Compliance in AI at the University of Oklahoma. In this assignment, you will analyze a dataset containing credit card default information. The dataset was obtained from the UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients. It was initially used for the study Yeh, I., & Lien, C. (2009). The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients. Expert Syst. Appl., 36, 2473-2480.

The variables in the dataset are:
- ID: unique identifier
- LIMIT_BAL: amount of credit given in NT dollars
- SEX: gender of the client (1=Male, 2=Female)
- EDUCATION: education level (1=Graduate School, 2=University, 3=High School, 4=Others, 5=Unknown, 6=Unknown)
- MARRIAGE: marital status (1=Married, 2=Single, 3=Others)
- AGE: age in of client in years
- PAY_0 to PAY_6: monthly repayment status from September 2005 to April 2005 in reverse order. Note that there is no PAY_1. This is a naming mistake by the authors. (-2=No consumption, -1=Paid in full, 0=Minimum payment, 1-8=Payment delay for x months, 9=Payment delay for nine or more months)
- BILL_AMT1 to BILL_AMT6: amount of bill statement in NT dollars from September 2005 to April 2005 in reverse order
- PAY_AMT1 to PAY_AMT6: amount of previous payment in NT dollars from September 2005 to April 2005 in reverse order
- default payment next month: whether the client made the next month payment - this is the label variable (1=Defaulted, 0=No Default)

In this notebook, _final\_project\_part\_3.ipynb_, you will read in the results of the baseline logistic regression model you saved to CSV in _final\_project\_part\_2.ipynb_. That file contains the true labels, predicted labels, predicted probabilities, and sensitive class values for the test split of the data. You will measure performance and fairness and then conduct post-processing to improve fairness.

### Import libraries

Import the libraries required for this notebook and set random seeds to stabilize results.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.metrics import confusion_matrix

import os, random

# Set global seeds for reproducibility
os.environ["PYTHONHASHSEED"] = "42"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
random.seed(42)
np.random.seed(42)


### Read in data from Part 2 and split by sensitive attribute

Read in _model\_results\_for\_postprocessing.csv into a pandas dataframe called df. Then split it into two new dataframes, _df\_0_, which contains males (A=0) and _df\_1_, which containes females (A=1). Make sure the new dataframes are copies of slices of the existing dataframe to ensure continutiy of changes.

In [ ]:
# Read in the results from part 2 and split into two groups

# BEGIN YOUR CODE HERE:


# END HERE

### Useful functions

You will use the functions in the next block in your analysis.  The first produces fairness metrics (this is the function from Part 2).  The second generates ROC curves and the AUROC for males and females.

In [ ]:
# Define the fairness evaluation function

def check_fairness(y_true, y_pred, sensitive_attr):
    """
    Evaluates a model's fairness by comparing performance across different groups.
    
    Parameters:
    y_true : Series/Array - The actual ground truth labels (0 or 1)
    y_pred : Series/Array - The model's binary predictions (0 or 1)
    sensitive_attr : Series/Array - The binary sensitive feature (e.g., Sex: 1=Male, 0=Female)
    """
    
    # Create a temporary dataframe to align predictions with actuals and sensitive labels
    results = pd.DataFrame({'y': y_true, 'y_hat': y_pred, 'A': sensitive_attr})
    
    # Split the results into two groups based on the sensitive attribute
    # Group 0 is often the historically 'unprivileged' or 'protected' group
    g0 = results[results['A'] == 0] 
    g1 = results[results['A'] == 1] 
    
    # Dictionary to store calculated metrics for each group
    stats = {}
    
    for name, group in [('Female', g0), ('Male', g1)]:
        # Calculate the Confusion Matrix components:
        # TN: True Negatives, FP: False Positives, FN: False Negatives, TP: True Positives
        # labels=[0,1] ensures the matrix is always 2x2 even if a group is missing a class
        tn, fp, fn, tp = confusion_matrix(group['y'], group['y_hat'], labels=[0, 1]).ravel()
        
        # 1. Demographic Parity (Statistical Independence)
        # Ratio of positive predictions regardless of the actual ground truth.
        # Ideally, P(Y_hat=1 | A=0) == P(Y_hat=1 | A=1)
        dp = group['y_hat'].mean()
        
        # 2. True Positive Rate (TPR / Sensitivity): Part of Error Rate Parity/Separation
        # How often does the model correctly predict 'High Income' for those who actually have it?
        # A gap here indicates the model 'misses' qualified candidates in one group more than the other.
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # 3. True Negative Rate (TNR / Specificity): Part of Error Rate Parity/Separation
        # How often does the model correctly predict 'Low Income' for those who actually have it?
        # A gap here indicates one group is being 'falsely accused' of being high-income more often.
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0

        # 4. False Positive Rate (FPR / Type I Error): Part of Error Rate Parity/Separation
        # How often does the model incorrectly predict 'High Income' for those who actually have it?
        # A gap here indicates one group is being 'falsely accused' of being high-income more often.
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

        # 5. Predictive Parity (Sufficiency)
        # If the model predicts 'High Income', how likely is it to be correct?
        # This checks if the 'meaning' of a positive prediction is the same across groups.
        pp = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Store metrics for this group
        stats[name] = {'DP': dp, 'TPR': tpr, 'TNR': tnr, 'FPR': fpr, 'PP': pp}

    # --- Print Results Table ---
    print(f"{'Metric':<22} | {'Female':<8} | {'Male':<8} | {'Difference':<10} | {'% Difference':<12}")
    print("-" * 65)
    
    # Iterate through the metrics to calculate and display the absolute difference
    for m in ['DP', 'TPR', 'TNR', 'FPR', 'PP']:
        # Rename keys for the final display
        display_names = {
            'DP': 'Demographic Parity',
            'TPR': 'True Positive Rate',
            'TNR': 'True Negative Rate',
            'FPR': 'False Positive Rate',
            'PP': 'Predictive Parity'
        }
        
        f_val = stats['Female'][m]
        m_val = stats['Male'][m]
        diff = abs(f_val - m_val)
        pdiff = abs(f_val - m_val)/f_val*100 if m_val > 0 else 0
        
        print(f"{display_names[m]:<22} | {f_val:.3f}   | {m_val:.3f}   | {diff:.3f}   | {pdiff:.1f}%")
        
    print("-" * 65)
    print("Note: 'Error Rate Parity' is satisfied if both TPR and FPR differences are near zero.")


def get_roc_data(y_true, y_probs):
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, thresholds, roc_auc
    
def plot_rocs(fpr0, tpr0, auc0, fpr1, tpr1, auc1, title="ROC Curves by Group"):
    plt.figure(figsize=(10, 6))
    plt.plot(fpr0, tpr0, label=f'A=0 (AUROC = {auc0:.3f})')
    plt.plot(fpr1, tpr1, label=f'A=1 (AUROC = {auc1:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.title(title)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.show()
    return

### Plot the ROCs for the baseline case and generate fairness metrics

You will generate the ROC curves for the baseline case by calling _get\_roc\_data()_ and _plot\_rocs()_. You will then generate fairness metrics for the base case like you did in Part 2.

Store the return from _get\_roc\_data()_ for males into _fpr0_, _tpr0_, _thresholds0_, and _auc0_. Do the same for females. Then use these objects in _plot\_rocs()_. Title your plot "Baseline ROC Curves."

In [ ]:
# Generate the ROC curves for both groups

# BEGIN YOUR CODE HERE:


# END HERE

In [ ]:
# Generate the fairness metrics for the baseline model

# ENTER YOUR CODE HERE:


# END HERE

### Optimize thresholds for fairness

You will use Hardt's algorithm to adjust the thresholds for the two classes so that we solve for Equal Odds (TPRs and FPRs are nearly the same). This is done in several steps.

### Step 1

Construct the feasible region from the two curves. This is done by taking the minimum TPR of the two curves for every point on the FPR axis. Then find the target (FPR, TPR) point -- the one that maximizes Youden's J.

In [ ]:
# Create an interpoloation grid to get TPRs at the same FPR for both groups
common_fpr = np.linspace(0, 1, 1000)
interp_tpr0 = np.interp(common_fpr, fpr0, tpr0)
interp_tpr1 = np.interp(common_fpr, fpr1, tpr1)

# The feasible region is the area under both curves
achievable_tpr = np.minimum(interp_tpr0, interp_tpr1)

# Find the point that maximizes Youden's J statistic (TPR - FPR) for both groups
idx = np.argmax(achievable_tpr - common_fpr)
target_fpr = common_fpr[idx]
target_tpr = achievable_tpr[idx]

print(f"Target Equalized Odds Point: FPR={target_fpr:.4f}, TPR={target_tpr:.4f}")

### Randomization

When one ROC curve is completely below another, there is no intersection between them. Hardt's solves this by brining down the upper ROC curve. In the code below, _achieveable\_tpr_ represents the inner envelope of both ROC curves. The alpha parameter is a scaling factor. If a class's curve is too high, we ignore its predictions with probability (1-alpha) and output 0. We then interpolate to get an approximization of the exact boundary.

In [ ]:
# This function applies the randomization to get near the target FPR and TPR for a given group

def apply_randomized_hardt(df_group, fpr_list, tpr_list, thresholds, target_fpr, target_tpr):
    """
    Finds the threshold on the ROC curve for the target FPR, 
    then scales the TPR down via randomization if necessary.
    """
    # Find the operating point on this group's ROC curve closest to target_fpr
    idx = np.searchsorted(fpr_list, target_fpr)
    
    # Linear interpolation for the threshold and TPR at the exact target_fpr
    if idx >= len(fpr_list): idx = len(fpr_list) - 1
    
    # Operating threshold for this group to hit target_fpr
    actual_thresh = thresholds[idx]
    tpr_on_curve = tpr_list[idx]
    
    # Calculate alpha: probability of 'keeping' the prediction to hit target_tpr
    # If target_tpr is 0.6 and this group's TPR at target_fpr is 0.8, alpha = 0.75
    alpha = target_tpr / tpr_on_curve if tpr_on_curve > 0 else 0
    
    # Generate randomized predictions
    probs = df_group['y_prob_base'].values
    deterministic_preds = (probs >= actual_thresh).astype(int)
    
    # Weighted coin flip: keep the prediction with probability alpha, else 0
    random_mask = np.random.rand(len(probs)) < alpha
    fair_preds = deterministic_preds * random_mask
    
    return fair_preds

### Get the new predictions

Call the _apply\_randomized\_hardt()_ function for both the male and female classes. Call the function with the target_fpr and target_tpr from above. Include the dataframe for each class and its fpr, tpr, and threshold lists. The results should be stored in a new variable called _y\_fair_. Your call will be of the form _df\_0['y\_fair'] = apply\_randomized\_hardt(...)_

In [ ]:
# Create the y_fair variables for both dataframes, df_0 and df_1.

# ENTER YOUR CODE HERE:


# END HERE

### Obtain the new results

Because Hardt's returns binary predictions rather than probabilities, we can't generate new ROC curves. Instead we plot the ...

In [ ]:
def get_final_metrics(df):
    tpr = np.mean(df['y_fair'][df['y_true'] == 1])
    fpr = np.mean(df['y_fair'][df['y_true'] == 0])
    # Binary AUROC for the point: (TPR + (1-FPR)) / 2
    b_auc = (tpr + (1 - fpr)) / 2
    return tpr, fpr, b_auc

t0, f0, a0 = get_final_metrics(df_0)
t1, f1, a1 = get_final_metrics(df_1)

print(f"\nA=0 Post-Processed: FPR={f0:.3f}, TPR={t0:.3f}, AUROC={a0:.3f}")
print(f"A=1 Post-Processed: FPR={f1:.3f}, TPR={t1:.3f}, AUROC={a1:.3f}")
print("FPR % Difference: {:.1f}%".format(abs(f0 - f1)/f0*100 if f0 > 0 else 0))
print("TPR % Difference: {:.1f}%".format(abs(t0 - t1)/t0*100 if t0 > 0 else 0))

# Plotting the "Intersection"
plt.figure(figsize=(10, 6))
plt.plot(fpr0, tpr0, label=f'A=0 ROC (Original AUC: {auc0:.2f})', alpha=0.4)
plt.plot(fpr1, tpr1, label=f'A=1 ROC (Original AUC: {auc1:.2f})', alpha=0.4)
plt.fill_between(common_fpr, achievable_tpr, color='gray', alpha=0.2, label='Feasible Equal Odds Area')

plt.scatter(f0, t0, color='blue', s=100, label='A=0 Final Point', zorder=5)
plt.scatter(f1, t1, color='orange', marker='x', s=100, label='A=1 Final Point', zorder=5)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Hardt Randomized Equalized Odds')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()